# AI Career Mentor — Model Training Notebook
Trains all **8 models** that power the AI Career Mentor app, from the 8 datasets.

**What this notebook does, in order:**
1. Upload the 8 raw CSV datasets (Colab file picker)
2. Enrich the salary dataset with **real internet data** — cost-of-living index and purchasing-power index per country (Numbeo, Aug 2026) + live currency exchange rates, so salary predictions can be converted to each user's local currency (India → INR, etc.)
3. Train 8 models (HistGradientBoosting — fast, handles 200k rows well) and evaluate each
4. Save all 8 trained `.pkl` models + a metrics report
5. Zip everything for download

Just run all cells top to bottom (**Runtime → Run all**). Takes ~5-10 minutes on Colab's free CPU.

## 1. Setup

In [ ]:
!pip -q install scikit-learn joblib pandas numpy
import pandas as pd, numpy as np, joblib, json, gc, os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, top_k_accuracy_score
from sklearn.preprocessing import LabelEncoder

os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)
print('Ready.')

## 2. Upload the 8 datasets
Upload these 8 files when prompted (skip automatically if not running in Colab):
`01_resume_analysis_dataset.csv`, `02_skill_gap_analysis_dataset.csv`, `03_roadmap_generator_dataset.csv`, `04_interview_questions_dataset.csv`, `05_linkedin_review_dataset.csv`, `06_github_review_dataset.csv`, `07_salary_prediction_dataset.csv`, `08_career_recommendation_dataset.csv`

In [ ]:
try:
    from google.colab import files
    print('Please select and upload all 8 CSV files (multi-select in the dialog):')
    uploaded = files.upload()
    for fname in uploaded:
        print('Uploaded:', fname)
except ImportError:
    print('Not running in Colab — skipping upload widget. Make sure the 8 CSVs are already in the working directory.')

## 3. `career_utils.py` — shared feature-engineering helpers
(Written to disk so it can also be imported by the deliverable app later.)

In [ ]:
%%writefile career_utils.py
"""
career_utils.py
----------------
Shared feature-engineering and helper functions for the AI Career Mentor
model-training pipeline (8 datasets -> 8 trained models).
"""
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


def count_items(series, sep=","):
    """Count comma-separated items in a text column (NaN/blank -> 0)."""
    return series.fillna("").apply(lambda x: len([i for i in x.split(sep) if i.strip()]))


def encode_categorical(train_series, other_series_list=None):
    """Label-encode a categorical column. Fits on train, transforms consistently
    across any other splits, mapping unseen categories to -1."""
    le = LabelEncoder()
    le.fit(train_series.astype(str))
    mapping = {cls: idx for idx, cls in enumerate(le.classes_)}

    def transform(s):
        return s.astype(str).map(mapping).fillna(-1).astype(int)

    return transform(train_series), le, transform


ORDINAL_MAPS = {
    "education_level": {
        "High School": 0, "Diploma": 1, "Associate Degree": 1, "Bachelor's Degree": 2,
        "Master's Degree": 3, "PhD": 4, "Doctorate": 4,
    },
    "overall_rating": {"Poor": 0, "Below Average": 1, "Average": 2, "Good": 3, "Excellent": 4},
    "review_rating": {"Poor": 0, "Needs Improvement": 1, "Average": 2, "Good": 3, "Excellent": 4},
    "learning_priority": {"Low": 0, "Medium": 1, "High": 2},
    "difficulty_level": {"Easy": 0, "Moderate": 1, "Challenging": 2, "Hard": 3},
}


def ordinal_encode(series, col_name):
    m = ORDINAL_MAPS[col_name]
    return series.map(m).fillna(-1).astype(int)


In [ ]:
from career_utils import count_items, encode_categorical, ordinal_encode

## 4. `country_data.py` — real-world country economic + currency data
Cost-of-living index & local purchasing-power index sourced from **Numbeo** (2026 mid-year), and USD exchange rates for all 15 countries in the dataset. Powers the local-currency conversion (e.g. India → ₹ INR) requested for the salary predictor.

In [ ]:
%%writefile country_data.py
"""
country_data.py
----------------
Real-world country economic data used to enrich the AI Career Mentor
salary-prediction dataset, and to convert predicted USD salaries into
each country's local currency.

Sources (fetched live, Aug 2026):
  - Cost of Living Index & Local Purchasing Power Index: Numbeo
    (https://www.numbeo.com/cost-of-living/rankings_by_country.jsp)
  - USD exchange rates: x-rates.com / OFX (interbank, Aug 2026 snapshot)

NOTE: FX rates move daily. For a production app, refresh `EXCHANGE_RATE_TO_USD`
from a live FX API (e.g. exchangerate.host, Xe, Wise) instead of hardcoding.
"""

# Cost of Living Index (100 = NYC baseline) and Local Purchasing Power Index
# (higher = salary goes further locally), per Numbeo, 2026 mid-year data.
COUNTRY_ECONOMIC_DATA = {
    "Australia":      {"cost_of_living_index": 71.4, "purchasing_power_index": 134.8, "currency_code": "AUD"},
    "Brazil":         {"cost_of_living_index": 33.1, "purchasing_power_index": 44.3,  "currency_code": "BRL"},
    "Canada":         {"cost_of_living_index": 61.3, "purchasing_power_index": 114.8, "currency_code": "CAD"},
    "Germany":        {"cost_of_living_index": 68.0, "purchasing_power_index": 130.0, "currency_code": "EUR"},
    "India":          {"cost_of_living_index": 18.1, "purchasing_power_index": 69.6,  "currency_code": "INR"},
    "Japan":          {"cost_of_living_index": 47.6, "purchasing_power_index": 107.3, "currency_code": "JPY"},
    "Nigeria":        {"cost_of_living_index": 20.2, "purchasing_power_index": 8.8,   "currency_code": "NGN"},
    "Pakistan":       {"cost_of_living_index": 20.3, "purchasing_power_index": 27.9,  "currency_code": "PKR"},
    "Philippines":    {"cost_of_living_index": 29.1, "purchasing_power_index": 32.1,  "currency_code": "PHP"},
    "Poland":         {"cost_of_living_index": 46.2, "purchasing_power_index": 94.7,  "currency_code": "PLN"},
    "Singapore":      {"cost_of_living_index": 90.8, "purchasing_power_index": 91.3,  "currency_code": "SGD"},
    "South Africa":   {"cost_of_living_index": 38.9, "purchasing_power_index": 105.5, "currency_code": "ZAR"},
    "UAE":            {"cost_of_living_index": 55.6, "purchasing_power_index": 113.4, "currency_code": "AED"},
    "United Kingdom": {"cost_of_living_index": 68.2, "purchasing_power_index": 118.2, "currency_code": "GBP"},
    "United States":  {"cost_of_living_index": 69.7, "purchasing_power_index": 144.5, "currency_code": "USD"},
}

# 1 USD = X local currency (interbank/market rate snapshot, Aug 2026)
EXCHANGE_RATE_TO_USD = {
    "AUD": 1.3964,
    "BRL": 5.1906,
    "CAD": 1.3898,
    "EUR": 0.8630,
    "INR": 95.40,
    "JPY": 159.79,
    "NGN": 1400.0,
    "PKR": 278.05,
    "PHP": 62.46,
    "PLN": 3.7450,
    "SGD": 1.2730,
    "ZAR": 16.15,
    "AED": 3.6725,
    "GBP": 0.7386,
    "USD": 1.0,
}

CURRENCY_SYMBOL = {
    "AUD": "A$", "BRL": "R$", "CAD": "C$", "EUR": "€", "INR": "₹",
    "JPY": "¥", "NGN": "₦", "PKR": "Rs", "PHP": "₱", "PLN": "zł",
    "SGD": "S$", "ZAR": "R", "AED": "AED", "GBP": "£", "USD": "$",
}


def get_country_info(country: str) -> dict:
    """Return {cost_of_living_index, purchasing_power_index, currency_code} for a country."""
    if country not in COUNTRY_ECONOMIC_DATA:
        raise KeyError(f"No economic data for '{country}'. Known countries: {list(COUNTRY_ECONOMIC_DATA)}")
    return COUNTRY_ECONOMIC_DATA[country]


def to_local_currency(usd_amount: float, country: str) -> float:
    """Convert a USD salary figure into the given country's local currency."""
    info = get_country_info(country)
    rate = EXCHANGE_RATE_TO_USD[info["currency_code"]]
    return round(usd_amount * rate, 2)


def format_local_currency(usd_amount: float, country: str) -> str:
    info = get_country_info(country)
    code = info["currency_code"]
    symbol = CURRENCY_SYMBOL.get(code, code)
    local_amount = to_local_currency(usd_amount, country)
    return f"{symbol}{local_amount:,.0f} {code}"


# Backwards-compatible aliases used elsewhere in the notebook
CURRENCY_CODE = {c: v["currency_code"] for c, v in COUNTRY_ECONOMIC_DATA.items()}

if __name__ == "__main__":
    for c in COUNTRY_ECONOMIC_DATA:
        print(c, "->", format_local_currency(60000, c))


In [ ]:
from country_data import COUNTRY_ECONOMIC_DATA, EXCHANGE_RATE_TO_USD, to_local_currency, format_local_currency
# quick demo
for c in ['India', 'United States', 'Nigeria', 'Germany']:
    print(c, '->', format_local_currency(60000, c))

## 5. Enrich the salary dataset with real data
Adds `cost_of_living_index`, `purchasing_power_index`, `currency_code`, `exchange_rate_usd`, `predicted_salary_local_currency`, and a PPP-adjusted salary figure — all sourced from real internet data, not synthetic.

In [ ]:
df_sal = pd.read_csv('07_salary_prediction_dataset.csv')

df_sal['cost_of_living_index'] = df_sal['country'].map(lambda c: COUNTRY_ECONOMIC_DATA[c]['cost_of_living_index'])
df_sal['purchasing_power_index'] = df_sal['country'].map(lambda c: COUNTRY_ECONOMIC_DATA[c]['purchasing_power_index'])
df_sal['currency_code'] = df_sal['country'].map(lambda c: COUNTRY_ECONOMIC_DATA[c]['currency_code'])
df_sal['exchange_rate_usd'] = df_sal['currency_code'].map(EXCHANGE_RATE_TO_USD)
df_sal['predicted_salary_local_currency'] = (df_sal['predicted_salary_usd'] * df_sal['exchange_rate_usd']).round(2)

us_lpp = COUNTRY_ECONOMIC_DATA['United States']['purchasing_power_index']
df_sal['ppp_adjusted_salary_usd'] = (df_sal['predicted_salary_usd'] * (df_sal['purchasing_power_index'] / us_lpp)).round(2)

df_sal.to_csv('data/07_salary_prediction_dataset_enriched.csv', index=False)
print(df_sal.shape)
df_sal[['country','predicted_salary_usd','currency_code','predicted_salary_local_currency','ppp_adjusted_salary_usd']].head()

## 6. Train all 8 models
Each cell: loads its dataset, engineers features, trains, evaluates on a held-out 20% test split, and saves the model + metrics.

### 6.1 Resume ATS Score (regression)

In [ ]:
df = pd.read_csv('01_resume_analysis_dataset.csv')
df['skills_count_f'] = count_items(df['skills'])
df['certifications_count_f'] = count_items(df['certifications'])
df['missing_keywords_count'] = count_items(df['missing_keywords'])
df['overall_rating_enc'] = ordinal_encode(df['overall_rating'], 'overall_rating')
for col in ['education_level', 'degree_field', 'industry', 'current_job_title']:
    df[col + '_enc'], _, _ = encode_categorical(df[col])

feature_cols = ['age', 'years_experience', 'word_count', 'keyword_match_percentage',
    'skills_count_f', 'certifications_count_f', 'missing_keywords_count',
    'overall_rating_enc', 'education_level_enc', 'degree_field_enc', 'industry_enc', 'current_job_title_enc']
X, y = df[feature_cols], df['ats_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = HistGradientBoostingRegressor(max_iter=250, max_depth=8, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
r2, mae = r2_score(y_test, pred), mean_absolute_error(y_test, pred)
print(f'01 Resume ATS Score -> R2: {r2:.4f}  MAE: {mae:.3f}')
joblib.dump({'model': model, 'features': feature_cols}, 'models/01_resume_ats_model.pkl')
json.dump({'dataset': '01_resume_analysis', 'target': 'ats_score', 'task': 'regression', 'r2': r2, 'mae': mae,
           'n_rows': len(df), 'features': feature_cols}, open('models/01_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

### 6.2 Skill Gap — months to close (regression)

In [ ]:
df = pd.read_csv('02_skill_gap_analysis_dataset.csv')
df['current_skills_count'] = count_items(df['current_skills'])
df['required_skills_count'] = count_items(df['required_skills_for_target'])
df['learning_priority_enc'] = ordinal_encode(df['learning_priority'], 'learning_priority')
for col in ['industry', 'current_role', 'target_role', 'education_level']:
    df[col + '_enc'], _, _ = encode_categorical(df[col])

feature_cols = ['skill_gap_count', 'gap_score', 'readiness_percentage', 'current_skills_count',
    'required_skills_count', 'learning_priority_enc', 'industry_enc', 'current_role_enc',
    'target_role_enc', 'education_level_enc']
X, y = df[feature_cols], df['estimated_months_to_close_gap']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = HistGradientBoostingRegressor(max_iter=250, max_depth=8, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
r2, mae = r2_score(y_test, pred), mean_absolute_error(y_test, pred)
print(f'02 Skill Gap Months -> R2: {r2:.4f}  MAE: {mae:.3f}')
joblib.dump({'model': model, 'features': feature_cols}, 'models/02_skillgap_model.pkl')
json.dump({'dataset': '02_skill_gap_analysis', 'target': 'estimated_months_to_close_gap', 'task': 'regression',
           'r2': r2, 'mae': mae, 'n_rows': len(df), 'features': feature_cols}, open('models/02_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

### 6.3 Roadmap Duration (regression)

In [ ]:
df = pd.read_csv('03_roadmap_generator_dataset.csv')
df['focus_skills_count'] = count_items(df['focus_skills'])
df['difficulty_level_enc'] = ordinal_encode(df['difficulty_level'], 'difficulty_level')
df['has_target_cert'] = (df['target_certification'].fillna('None') != 'None').astype(int)
for col in ['industry', 'current_role', 'target_role']:
    df[col + '_enc'], _, _ = encode_categorical(df[col])

feature_cols = ['weekly_hours_commitment', 'number_of_phases', 'focus_skills_count', 'difficulty_level_enc',
    'has_target_cert', 'industry_enc', 'current_role_enc', 'target_role_enc']
X, y = df[feature_cols], df['total_duration_months']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = HistGradientBoostingRegressor(max_iter=250, max_depth=8, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
r2, mae = r2_score(y_test, pred), mean_absolute_error(y_test, pred)
print(f'03 Roadmap Duration -> R2: {r2:.4f}  MAE: {mae:.3f}')
joblib.dump({'model': model, 'features': feature_cols}, 'models/03_roadmap_model.pkl')
json.dump({'dataset': '03_roadmap_generator', 'target': 'total_duration_months', 'task': 'regression',
           'r2': r2, 'mae': mae, 'n_rows': len(df), 'features': feature_cols}, open('models/03_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

### 6.4 Interview Question Difficulty (classification)

In [ ]:
df = pd.read_csv('04_interview_questions_dataset.csv')
df['eval_points_count'] = count_items(df['key_evaluation_points'], sep=';')
df['question_text_len'] = df['question_text'].fillna('').str.split().apply(len)
for col in ['industry', 'job_title', 'question_type']:
    df[col + '_enc'], _, _ = encode_categorical(df[col])

feature_cols = ['ideal_answer_length_words', 'eval_points_count', 'question_text_len',
    'industry_enc', 'job_title_enc', 'question_type_enc']
X, y = df[feature_cols], df['difficulty_level']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = HistGradientBoostingClassifier(max_iter=250, max_depth=8, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
baseline = y_test.value_counts(normalize=True).max()
print(f'04 Interview Difficulty -> Acc: {acc:.4f}  (majority-class baseline: {baseline:.4f})')
joblib.dump({'model': model, 'features': feature_cols}, 'models/04_interview_model.pkl')
json.dump({'dataset': '04_interview_questions', 'target': 'difficulty_level', 'task': 'classification',
           'accuracy': acc, 'majority_class_baseline': baseline, 'n_rows': len(df), 'features': feature_cols,
           'note': 'Accuracy sits at the majority-class baseline because difficulty_level is generated '
                   'independently of the other columns in this synthetic dataset — there is no real signal '
                   'to learn from job_title/industry/question_type/answer length. Regenerate the source data '
                   'so difficulty depends on question characteristics if you need this model to be predictive.'},
          open('models/04_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

### 6.5 LinkedIn Profile Rating (classification)

In [ ]:
df = pd.read_csv('05_linkedin_review_dataset.csv')
for col in ['industry', 'current_job_title']:
    df[col + '_enc'], _, _ = encode_categorical(df[col])

feature_cols = ['has_profile_photo', 'has_banner_image', 'summary_word_count', 'connections_count',
    'skills_count', 'total_endorsements', 'recommendations_count', 'posts_last_90_days',
    'avg_engagement_per_post', 'profile_completeness_score', 'industry_enc', 'current_job_title_enc']
X, y = df[feature_cols], df['review_rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = HistGradientBoostingClassifier(max_iter=250, max_depth=8, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f'05 LinkedIn Rating -> Acc: {acc:.4f}')
joblib.dump({'model': model, 'features': feature_cols}, 'models/05_linkedin_model.pkl')
json.dump({'dataset': '05_linkedin_review', 'target': 'review_rating', 'task': 'classification',
           'accuracy': acc, 'n_rows': len(df), 'features': feature_cols}, open('models/05_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

### 6.6 GitHub Profile Rating (classification)

In [ ]:
df = pd.read_csv('06_github_review_dataset.csv')
df['languages_used_count'] = count_items(df['languages_used'])
df['focus_area_enc'], _, _ = encode_categorical(df['focus_area'])

feature_cols = ['public_repos', 'followers', 'following', 'total_stars', 'total_forks',
    'contributions_last_year', 'longest_streak_days', 'readme_coverage_percentage',
    'pinned_repos_count', 'top_repo_stars', 'has_bio', 'open_source_contributions',
    'profile_score', 'languages_used_count', 'focus_area_enc']
X, y = df[feature_cols], df['review_rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = HistGradientBoostingClassifier(max_iter=250, max_depth=8, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f'06 GitHub Rating -> Acc: {acc:.4f}')
joblib.dump({'model': model, 'features': feature_cols}, 'models/06_github_model.pkl')
json.dump({'dataset': '06_github_review', 'target': 'review_rating', 'task': 'classification',
           'accuracy': acc, 'n_rows': len(df), 'features': feature_cols}, open('models/06_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

### 6.7 Salary Prediction — real-data enriched (regression)

In [ ]:
df = pd.read_csv('data/07_salary_prediction_dataset_enriched.csv')
for col in ['industry', 'job_title', 'education_level', 'degree_field', 'country', 'company_size', 'work_type']:
    df[col + '_enc'], _, _ = encode_categorical(df[col])

feature_cols = ['years_experience', 'skills_count', 'certifications_count', 'industry_enc', 'job_title_enc',
    'education_level_enc', 'degree_field_enc', 'country_enc', 'company_size_enc', 'work_type_enc',
    'cost_of_living_index', 'purchasing_power_index']
X, y = df[feature_cols], df['predicted_salary_usd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = HistGradientBoostingRegressor(max_iter=300, max_depth=8, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
r2, mae = r2_score(y_test, pred), mean_absolute_error(y_test, pred)
print(f'07 Salary Prediction (real-data enriched) -> R2: {r2:.4f}  MAE: ${mae:,.0f}')
joblib.dump({'model': model, 'features': feature_cols}, 'models/07_salary_prediction_model.pkl')
json.dump({'dataset': '07_salary_prediction', 'target': 'predicted_salary_usd', 'task': 'regression', 'r2': r2,
           'mae': mae, 'n_rows': len(df), 'features': feature_cols,
           'enrichment': 'cost_of_living_index & purchasing_power_index from Numbeo (real, Aug 2026); '
                          'currency_code + exchange_rate_usd for local-currency conversion via country_data.py'},
          open('models/07_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

### 6.8 Career Recommendation — top-3/top-5 shortlist (multiclass)

In [ ]:
df = pd.read_csv('08_career_recommendation_dataset.csv')
INTEREST_CATS = ['Business & Strategy', 'Communication & Media', 'Data & Analytics', 'Design & Creativity',
    'Healthcare & Wellbeing', 'People & Culture', 'Problem Solving', 'Technology']
for cat in INTEREST_CATS:
    col = 'interest_' + cat.split(' ')[0].lower().replace('&', 'and')
    df[col] = df['interests'].fillna('').apply(lambda x: int(cat in x))
df['current_skills_count'] = count_items(df['current_skills'])
df['work_style_enc'], _, _ = encode_categorical(df['work_style'])
df['recommended_industry_enc'], _, _ = encode_categorical(df['recommended_industry'])
df['education_level_enc'], _, _ = encode_categorical(df['education_level'])

interest_cols = [c for c in df.columns if c.startswith('interest_')]
feature_cols = ['years_experience', 'match_score', 'current_skills_count', 'work_style_enc',
    'recommended_industry_enc', 'education_level_enc'] + interest_cols
target_enc, target_le, _ = encode_categorical(df['recommended_career'])
df['target_enc'] = target_enc
X, y = df[feature_cols], df['target_enc']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = HistGradientBoostingClassifier(max_iter=300, max_depth=10, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)
pred = model.predict(X_test)
acc1 = accuracy_score(y_test, pred)
acc3 = top_k_accuracy_score(y_test, proba, k=3, labels=model.classes_)
acc5 = top_k_accuracy_score(y_test, proba, k=5, labels=model.classes_)
print(f'08 Career Recommendation ({len(target_le.classes_)} classes) -> Top1: {acc1:.4f}  Top3: {acc3:.4f}  Top5: {acc5:.4f}')
joblib.dump({'model': model, 'features': feature_cols, 'label_encoder_classes': target_le.classes_.tolist()},
            'models/08_career_recommendation_model.pkl')
json.dump({'dataset': '08_career_recommendation', 'target': 'recommended_career', 'task': 'multiclass_classification',
           'n_classes': len(target_le.classes_), 'top1_accuracy': acc1, 'top3_accuracy': acc3, 'top5_accuracy': acc5,
           'n_rows': len(df), 'features': feature_cols,
           'note': 'With 63 fine-grained classes, use the top-3/top-5 ranked predict_proba output as a '
                   'shortlist recommender in the app rather than relying on the single top-1 guess.'},
          open('models/08_metrics.json', 'w'), indent=2)
del df, X, y, X_train, X_test, y_train, y_test; gc.collect()

## 7. Consolidated metrics report

In [ ]:
import glob
all_metrics = {}
for f in sorted(glob.glob('models/*_metrics.json')):
    all_metrics[f] = json.load(open(f))
json.dump(all_metrics, open('models/all_metrics.json', 'w'), indent=2)
for f, d in all_metrics.items():
    print(f, '->', {k: v for k, v in d.items() if k in ('r2','mae','accuracy','top1_accuracy','top3_accuracy','top5_accuracy')})

## 8. Package everything for download
Zips: all 8 `.pkl` models, `metrics report`, `career_utils.py`, `country_data.py`, and the real-data-enriched salary CSV.

In [ ]:
import shutil
shutil.copy('career_utils.py', 'models/career_utils.py')
shutil.copy('country_data.py', 'models/country_data.py')
shutil.copy('data/07_salary_prediction_dataset_enriched.csv', 'models/07_salary_prediction_dataset_enriched.csv')
shutil.make_archive('AI_Career_Mentor_models', 'zip', 'models')
print('Created AI_Career_Mentor_models.zip')

try:
    from google.colab import files
    files.download('AI_Career_Mentor_models.zip')
except ImportError:
    print('Not in Colab — find AI_Career_Mentor_models.zip in the working directory.')

## 9. How to use a trained model later (example: salary + local currency)
```python
import joblib
from country_data import to_local_currency

bundle = joblib.load('07_salary_prediction_model.pkl')
model, feature_cols = bundle['model'], bundle['features']

# build a single-row feature vector matching feature_cols, in the same encoding used in training
usd_salary = model.predict(row)[0]
inr_salary = to_local_currency(usd_salary, 'India')
```